## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Fortgeschrittene
Aufgabe 12.1: Conditional GAN (cGAN) – Gezielte Bildgenerierung

Lernziel: cGAN für klassen-konditionierte Generierung
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32')
X_train = (X_train - 127.5) / 127.5
y_train_oh = keras.utils.to_categorical(y_train, 10)

LATENT_DIM  = 100
N_KLASSEN   = 10
BATCH_SIZE  = 256

# Conditional Generator
def cond_generator():
    z_input     = keras.Input(shape=(LATENT_DIM,))
    label_input = keras.Input(shape=(N_KLASSEN,))
    x = keras.layers.Concatenate()([z_input, label_input])
    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.Dense(512, activation='relu')(x)
    ausgabe = keras.layers.Dense(784, activation='tanh')(x)
    return keras.Model([z_input, label_input], ausgabe, name='cGenerator')

# Conditional Diskriminator
def cond_diskriminator():
    bild_input  = keras.Input(shape=(784,))
    label_input = keras.Input(shape=(N_KLASSEN,))
    x = keras.layers.Concatenate()([bild_input, label_input])
    x = keras.layers.Dense(512, activation='leaky_relu')(x)
    x = keras.layers.Dense(256, activation='leaky_relu')(x)
    ausgabe = keras.layers.Dense(1)(x)
    return keras.Model([bild_input, label_input], ausgabe, name='cDiskriminator')

generator     = cond_generator()
diskriminator = cond_diskriminator()

bce   = keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_schritt(bilder, labels):
    batch_n = tf.shape(bilder)[0]
    z = tf.random.normal((batch_n, LATENT_DIM))
    
    with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
        fake = generator([z, labels], training=True)
        d_real = diskriminator([bilder, labels], training=True)
        d_fake = diskriminator([fake, labels], training=True)
        
        d_loss = bce(tf.ones_like(d_real), d_real) + bce(tf.zeros_like(d_fake), d_fake)
        g_loss = bce(tf.ones_like(d_fake), d_fake)
    
    d_grads = d_tape.gradient(d_loss, diskriminator.trainable_variables)
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, diskriminator.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))
    return d_loss, g_loss

dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train_oh.astype('float32'))
).shuffle(60000).batch(BATCH_SIZE)

EPOCHEN = 10
print("Training Conditional GAN...")
for epoche in range(EPOCHEN):
    d_l_sum = g_l_sum = n = 0
    for bilder, labels in dataset:
        d_l, g_l = train_schritt(bilder, labels)
        d_l_sum += float(d_l); g_l_sum += float(g_l); n += 1
    print(f"Epoche {epoche+1:2d}: D={d_l_sum/n:.4f} G={g_l_sum/n:.4f}")

# Gezielte Generierung: eine Reihe pro Ziffer
fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for klasse in range(10):
    label_oh = np.zeros((10, 10), dtype='float32')
    label_oh[:, klasse] = 1.0
    z = np.random.randn(10, LATENT_DIM).astype('float32')
    gen = generator([z, label_oh], training=False).numpy()
    gen = (gen + 1) / 2.0
    
    for j in range(10):
        axes[klasse, j].imshow(gen[j].reshape(28, 28), cmap='gray')
        axes[klasse, j].axis('off')
    axes[klasse, 0].set_ylabel(str(klasse), rotation=0, labelpad=15, fontsize=12)

plt.suptitle('Conditional GAN: Gezielte Zifferngenerierung (Zeilen=Klassen)')
plt.tight_layout()
plt.savefig('cgan_gezielte_generierung.png', dpi=150)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Fortgeschrittene
Aufgabe 12.2: Wasserstein GAN (WGAN) – Stabiles Training

Lernziel: WGAN als stabilere Alternative zu Standard-GAN
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0 * 2 - 1

LATENT_DIM = 64
BATCH_SIZE  = 256
CLIP_VALUE  = 0.01
N_CRITIC    = 5

def generator():
    return keras.Sequential([
        keras.layers.Dense(256, activation='relu', input_shape=(LATENT_DIM,)),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dense(784, activation='tanh')
    ], name='WGANGenerator')

def critic():
    """Kein Sigmoid – WGAN Critic gibt reelle Zahlen aus."""
    return keras.Sequential([
        keras.layers.Dense(512, activation='leaky_relu', input_shape=(784,)),
        keras.layers.Dense(256, activation='leaky_relu'),
        keras.layers.Dense(1)  # Kein Sigmoid!
    ], name='WGANCritic')

G = generator()
C = critic()

g_opt = keras.optimizers.RMSprop(0.00005)
c_opt = keras.optimizers.RMSprop(0.00005)

def wasserstein_loss_real(y_pred):
    return -tf.reduce_mean(y_pred)

def wasserstein_loss_fake(y_pred):
    return tf.reduce_mean(y_pred)

dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(BATCH_SIZE)

EPOCHEN = 15
g_losses_w, c_losses_w = [], []
std_g_loss, std_dcgan = [], []  # Vergleich

print("Training WGAN...")
for epoche in range(EPOCHEN):
    g_l_sum = c_l_sum = 0; n = 0
    
    for real_batch in dataset:
        batch_n = tf.shape(real_batch)[0]
        
        # Critic N_CRITIC Mal trainieren
        for _ in range(N_CRITIC):
            z = tf.random.normal((batch_n, LATENT_DIM))
            with tf.GradientTape() as c_tape:
                fake  = G(z, training=True)
                c_real = C(real_batch, training=True)
                c_fake = C(fake, training=True)
                c_loss = wasserstein_loss_real(c_real) + wasserstein_loss_fake(c_fake)
            c_grads = c_tape.gradient(c_loss, C.trainable_variables)
            c_opt.apply_gradients(zip(c_grads, C.trainable_variables))
            
            # Gewichts-Clipping (WGAN-Bedingung)
            for var in C.trainable_variables:
                var.assign(tf.clip_by_value(var, -CLIP_VALUE, CLIP_VALUE))
        
        # Generator 1x trainieren
        z = tf.random.normal((batch_n, LATENT_DIM))
        with tf.GradientTape() as g_tape:
            fake = G(z, training=True)
            g_loss = -tf.reduce_mean(C(fake, training=True))
        g_grads = g_tape.gradient(g_loss, G.trainable_variables)
        g_opt.apply_gradients(zip(g_grads, G.trainable_variables))
        
        g_l_sum += float(g_loss); c_l_sum += float(c_loss); n += 1
    
    g_losses_w.append(g_l_sum/n)
    c_losses_w.append(c_l_sum/n)
    print(f"Epoche {epoche+1:2d}: Wasserstein: C={c_l_sum/n:.4f} G={g_l_sum/n:.4f}")

# Generierte Bilder
z = tf.random.normal((16, LATENT_DIM))
generiert = (G(z) + 1) / 2.0

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generiert[i].numpy().reshape(28,28), cmap='gray')
    ax.axis('off')
plt.suptitle('WGAN Generierte Bilder')
plt.tight_layout()
plt.savefig('wgan_generiert.png', dpi=150)
plt.show()

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(c_losses_w, label='Critic (WGAN)', color='#FF6B6B')
plt.title('Wasserstein Distanz (Critic Loss)')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(g_losses_w, label='Generator (WGAN)', color='#00E6FF')
plt.title('Generator Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.suptitle('WGAN Training Stability')
plt.tight_layout()
plt.savefig('wgan_losses.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 12: Generative Adversarial Networks (GAN)
Niveau: Fortgeschrittene
Aufgabe 12.3: GAN Trainings-Tricks und Mode Collapse Diagnose

Lernziel: Typische GAN-Probleme erkennen und beheben
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), _ = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0 * 2 - 1

LATENT_DIM = 64

def erstelle_gan(lr_g=2e-4, lr_d=2e-4, label_smoothing=0.9):
    g = keras.Sequential([
        keras.layers.Dense(256, activation='relu', input_shape=(LATENT_DIM,)),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(784, activation='tanh')
    ])
    
    d = keras.Sequential([
        keras.layers.Dense(512, activation='leaky_relu', input_shape=(784,)),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation='leaky_relu'),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    return g, d, keras.optimizers.Adam(lr_g, beta_1=0.5), keras.optimizers.Adam(lr_d, beta_1=0.5)

# Trainiere mit verschiedenen Tricks
def trainiere_gan(label_smooth, epochen=10):
    G, D, g_opt, d_opt = erstelle_gan(label_smoothing=label_smooth)
    bce = keras.losses.BinaryCrossentropy()
    dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(128)
    
    d_losses, g_losses = [], []
    
    for epoche in range(epochen):
        for real_batch in dataset:
            n = tf.shape(real_batch)[0]
            z = tf.random.normal((n, LATENT_DIM))
            
            # One-sided label smoothing
            real_labels = tf.ones((n, 1)) * label_smooth
            fake_labels = tf.zeros((n, 1))
            
            with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
                fake  = G(z, training=True)
                d_r   = D(real_batch, training=True)
                d_f   = D(fake, training=True)
                d_l   = bce(real_labels, d_r) + bce(fake_labels, d_f)
                g_l   = bce(tf.ones((n, 1)), d_f)
            
            d_grads = d_tape.gradient(d_l, D.trainable_variables)
            g_grads = g_tape.gradient(g_l, G.trainable_variables)
            d_opt.apply_gradients(zip(d_grads, D.trainable_variables))
            g_opt.apply_gradients(zip(g_grads, G.trainable_variables))
        
        d_losses.append(float(d_l))
        g_losses.append(float(g_l))
    
    return G, d_losses, g_losses

print("Training mit Label Smoothing=1.0 (kein Smoothing)...")
G1, d_l1, g_l1 = trainiere_gan(1.0)
print("Training mit Label Smoothing=0.9...")
G2, d_l2, g_l2 = trainiere_gan(0.9)

# Diversity Score (Approximation für Mode Collapse)
def diversity_score(generator, n=1000):
    z = np.random.randn(n, LATENT_DIM).astype('float32')
    samples = generator(z, training=False).numpy()
    return np.mean(np.std(samples, axis=0))

print(f"\nDiversity Score (ohne Smoothing): {diversity_score(G1):.4f}")
print(f"Diversity Score (mit Smoothing):  {diversity_score(G2):.4f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(d_l1, label='D Loss', color='#FF6B6B')
axes[0, 0].plot(g_l1, label='G Loss', color='#00E6FF')
axes[0, 0].set_title('Ohne Label Smoothing')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(d_l2, label='D Loss', color='#FF6B6B')
axes[0, 1].plot(g_l2, label='G Loss', color='#00E6FF')
axes[0, 1].set_title('Mit Label Smoothing (0.9)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

z = np.random.randn(8, LATENT_DIM).astype('float32')
for i, (G, ax_row) in enumerate([(G1, axes[1, 0]), (G2, axes[1, 1])]):
    gen = (G(z, training=False).numpy() + 1) / 2
    for j in range(4):
        inset = ax_row.inset_axes([j*0.25, 0, 0.25, 1])
        inset.imshow(gen[j].reshape(28,28), cmap='gray')
        inset.axis('off')
    ax_row.set_title(['Ohne Smoothing', 'Mit Smoothing'][i])
    ax_row.axis('off')

plt.suptitle('GAN Trainings-Tricks: Label Smoothing Effekt')
plt.tight_layout()
plt.savefig('gan_training_tricks.png', dpi=150)
plt.show()
